# Multi-Agent System: Planner → Executor → Reviewer

This notebook builds a guarded multi-agent system with:

1. **Environment Setup** — Local LLM via Ollama (`llama3`) or free-tier Groq API.
2. **Agent Graph** — A 3-node graph (Planner → Executor → Reviewer) built with **LangGraph**, where each node's reasoning is delegated to a **CrewAI** agent.
3. **Loop & Budget Guardrails** — An `execution_count` in the Graph State with a conditional edge that terminates and returns a safe fallback response once `execution_count > 3`.
4. **Output Validation & Memory** — Every agent response is parsed into a strict **Pydantic** schema, and **Mem0** (backed by a local SQLite database) persists user state across sessions so we don't need to resend raw context tokens.

## 1. Environment Setup

Choose **one** of the two backends:

- **Ollama (local, free)** — install [Ollama](https://ollama.com), then run `ollama run llama3` once in a terminal to pull the model. No API key needed. (Not available on hosted runtimes like Colab unless you run Ollama on a separate machine and expose it.)
- **Groq (cloud, free-tier)** — create a key at the [Groq Console](https://console.groq.com/keys) and set it as `GROQ_API_KEY` below. **Use this on Colab.**

The backend is auto-detected (Ollama if reachable at `localhost:11434`, otherwise Groq). To force one, set `os.environ["LLM_BACKEND"] = "ollama"` or `"groq"` before running the next cell.

In [ ]:
%pip install -q langgraph "crewai[litellm]" pydantic mem0ai chromadb sentence-transformers ollama

> ⚠️ **After running the install cell above for the first time (or after changing it), restart the runtime/kernel** (Colab: *Runtime → Restart session*) before running any other cell. Installing/upgrading packages like `chromadb`/`sentence-transformers` can pull in a newer `Pillow`, but the already-running Python process keeps the old one loaded in memory — causing errors like `ImportError: cannot import name '_Ink' from 'PIL._typing'`. Restarting clears that stale state.

In [ ]:
import os
import socket

def _ollama_reachable(host: str = "localhost", port: int = 11434, timeout: float = 1.0) -> bool:
    """Colab/hosted runtimes have no local Ollama server, so probe before assuming it's usable."""
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False

# --- Choose your backend ---
# Set os.environ["LLM_BACKEND"] = "ollama" or "groq" yourself to force one; otherwise it's auto-detected.

_requested_backend = os.environ.get("LLM_BACKEND")print(f"Using LLM backend: {LLM_BACKEND}")

LLM_BACKEND = _requested_backend or ("ollama" if _ollama_reachable() else "groq")

OLLAMA_MODEL = "llama3"    )

GROQ_MODEL = "llama-3.1-8b-instant"        "(this is required on hosted runtimes like Colab, which have no local Ollama server)."

        "Run `ollama serve` and `ollama pull llama3` first, or set LLM_BACKEND='groq' with a GROQ_API_KEY instead "

if LLM_BACKEND == "groq":        "LLM_BACKEND='ollama' but no local Ollama server was found at localhost:11434. "

    os.environ.setdefault("GROQ_API_KEY", "<PASTE_YOUR_GROQ_API_KEY_HERE>")    raise RuntimeError(

    assert os.environ["GROQ_API_KEY"] and "PASTE" not in os.environ["GROQ_API_KEY"], \elif not _ollama_reachable():
        "Set a real GROQ_API_KEY (https://console.groq.com/keys) or run Ollama locally and set LLM_BACKEND='ollama'."

In [ ]:
from crewai import LLM

def get_chat_model() -> LLM:
    """Returns a CrewAI LLM (provider/model string, per CrewAI's LiteLLM-backed convention) for the configured backend."""
    if LLM_BACKEND == "groq":
        return LLM(model=f"groq/{GROQ_MODEL}", temperature=0)
    else:
        return LLM(model=f"ollama/{OLLAMA_MODEL}", temperature=0, base_url="http://localhost:11434")

llm = get_chat_model()
print(llm.model)

## 2. Output Validation Schemas (Pydantic)

Every node must return a strictly-typed object. We parse raw LLM text into these schemas so the graph always receives deterministic, validated data instead of free-form strings.

In [ ]:
import json
import re
from typing import List, Optional
from pydantic import BaseModel, Field, ValidationError


class PlannerOutput(BaseModel):
    """Strict schema the Planner agent must produce."""
    steps: List[str] = Field(..., description="Ordered list of concrete steps to solve the task.")
    rationale: str = Field(..., description="Short reasoning for the chosen plan.")


class ExecutorOutput(BaseModel):
    """Strict schema the Executor agent must produce."""
    result: str = Field(..., description="The concrete output produced by executing the plan.")
    confidence: float = Field(..., ge=0.0, le=1.0, description="Executor's self-reported confidence.")


class ReviewerOutput(BaseModel):
    """Strict schema the Reviewer agent must produce."""
    approved: bool = Field(..., description="Whether the executor's result is acceptable.")
    feedback: str = Field(..., description="Feedback used to re-plan if not approved.")


def _extract_json(text: str) -> dict:
    """Pulls the first JSON object out of an LLM response, tolerating markdown code fences."""
    fenced = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    candidate = fenced.group(1) if fenced else text
    match = re.search(r"\{.*\}", candidate, re.DOTALL)
    if not match:
        raise ValueError(f"No JSON object found in LLM output: {text!r}")
    return json.loads(match.group(0))


def parse_strict(text: str, schema: type[BaseModel]) -> BaseModel:
    """Parses raw LLM text into a Pydantic schema, raising on any deviation (deterministic parsing)."""
    data = _extract_json(text)
    try:
        return schema.model_validate(data)
    except ValidationError as e:
        raise ValueError(f"Output failed {schema.__name__} validation: {e}") from e

## 3. Persistent Memory (Mem0 + local SQLite)

`mem0` stores condensed *facts* about a user (not raw transcripts). Its vector index and conversation history are both persisted to a local `./mem0_data` folder (SQLite-backed), so a new session can recall prior state without resending the full chat history.

In [ ]:
import os
from mem0 import Memory

os.makedirs("./mem0_data", exist_ok=True)

mem0_config = {
    "vector_store": {
        "provider": "chroma",
        "config": {"collection_name": "mas_memory", "path": "./mem0_data/chroma"},
    },
    # mem0 also keeps its own conversation-history index in a local SQLite
    # file (./mem0_data/history.db) automatically.
    "history_db_path": "./mem0_data/history.db",
    "llm": {
        "provider": "groq" if LLM_BACKEND == "groq" else "ollama",
        "config": {"model": GROQ_MODEL if LLM_BACKEND == "groq" else OLLAMA_MODEL, "temperature": 0},
    },
    # Ollama's embedder needs a local `ollama serve` process; on hosted runtimes (e.g. Colab)
    # or the "groq" backend there's no such server, so fall back to a local, server-free
    # sentence-transformers embedding model instead.
    "embedder": (
        {"provider": "ollama", "config": {"model": "nomic-embed-text"}}
        if LLM_BACKEND == "ollama"
        else {"provider": "huggingface", "config": {"model": "sentence-transformers/all-MiniLM-L6-v2"}}
    ),
}

memory = Memory.from_config(mem0_config)
print("Mem0 initialized. Backing store: ./mem0_data")

In [ ]:
def remember(user_id: str, role: str, content: str) -> None:
    """Distills a message into Mem0 instead of persisting the raw transcript."""
    memory.add([{"role": role, "content": content}], user_id=user_id)


def recall(user_id: str, query: str, limit: int = 5) -> str:
    """Retrieves condensed prior context relevant to `query`, to prepend instead of full history."""
    hits = memory.search(query=query, user_id=user_id, limit=limit)
    results = hits["results"] if isinstance(hits, dict) else hits  # mem0 versions differ: dict-wrapped or plain list
    facts = [h["memory"] for h in results]
    return "\n".join(f"- {f}" for f in facts) if facts else "(no prior context)"

## 4. CrewAI Agents (Planner / Executor / Reviewer)

Each LangGraph node delegates its reasoning to a dedicated CrewAI `Agent` + one-off `Task`, all sharing the same underlying LLM.

In [ ]:
from crewai import Agent, Task, Crew, Process

planner_agent = Agent(
    role="Planner",
    goal="Break the user's task into a short, ordered list of concrete steps.",
    backstory="A meticulous project planner who never writes prose, only structured JSON plans.",
    llm=llm,
    verbose=False,
)

executor_agent = Agent(
    role="Executor",
    goal="Carry out the given plan and produce a concrete result.",
    backstory="A pragmatic doer who executes plans literally and reports a confidence score.",
    llm=llm,
    verbose=False,
)

reviewer_agent = Agent(
    role="Reviewer",
    goal="Critically check whether the executor's result actually satisfies the original task.",
    backstory="A strict QA reviewer who rejects incomplete or incorrect results with actionable feedback.",
    llm=llm,
    verbose=False,
)


def run_crew_task(agent: Agent, description: str, expected_output: str) -> str:
    """Runs a single CrewAI agent on a one-off task and returns its raw text output."""
    task = Task(description=description, expected_output=expected_output, agent=agent)
    crew = Crew(agents=[agent], tasks=[task], process=Process.sequential, verbose=False)
    return str(crew.kickoff())

## 5. Graph State & Loop/Budget Guardrails

`execution_count` increments every time the loop goes Planner → Executor → Reviewer. If the Reviewer rejects the result and `execution_count > 3`, the conditional edge routes straight to a `fallback` node instead of looping again.

In [ ]:
from typing import TypedDict

MAX_EXECUTIONS = 3  # hard budget: loop terminates once execution_count > this value

class GraphState(TypedDict):
    user_id: str
    task: str
    plan: Optional[PlannerOutput]
    execution: Optional[ExecutorOutput]
    review: Optional[ReviewerOutput]
    execution_count: int
    final_response: Optional[str]

In [ ]:
def planner_node(state: GraphState) -> GraphState:
    context = recall(state["user_id"], state["task"])
    feedback = state["review"].feedback if state.get("review") else "none yet"
    raw = run_crew_task(
        planner_agent,
        description=(
            f"Task: {state['task']}\n"
            f"Relevant prior context (condensed, not raw history):\n{context}\n"
            f"Reviewer feedback from last attempt: {feedback}\n"
            "Respond with ONLY a JSON object matching this schema: "
            '{"steps": ["..."], "rationale": "..."}'
        ),
        expected_output="A single JSON object with keys 'steps' and 'rationale'.",
    )
    plan = parse_strict(raw, PlannerOutput)
    remember(state["user_id"], "assistant", f"Plan: {plan.rationale}")
    return {**state, "plan": plan}


def executor_node(state: GraphState) -> GraphState:
    raw = run_crew_task(
        executor_agent,
        description=(
            f"Execute this plan for task '{state['task']}':\n"
            f"{state['plan'].steps}\n"
            "Respond with ONLY a JSON object matching this schema: "
            '{"result": "...", "confidence": 0.0}'
        ),
        expected_output="A single JSON object with keys 'result' and 'confidence'.",
    )
    execution = parse_strict(raw, ExecutorOutput)
    new_count = state["execution_count"] + 1
    remember(state["user_id"], "assistant", f"Execution result: {execution.result}")
    return {**state, "execution": execution, "execution_count": new_count}


def reviewer_node(state: GraphState) -> GraphState:
    raw = run_crew_task(
        reviewer_agent,
        description=(
            f"Original task: {state['task']}\n"
            f"Executor result: {state['execution'].result}\n"
            f"Executor confidence: {state['execution'].confidence}\n"
            "Decide if this fully satisfies the task. Respond with ONLY a JSON object: "
            '{"approved": true/false, "feedback": "..."}'
        ),
        expected_output="A single JSON object with keys 'approved' and 'feedback'.",
    )
    review = parse_strict(raw, ReviewerOutput)
    remember(state["user_id"], "assistant", f"Review: approved={review.approved}, {review.feedback}")
    return {**state, "review": review}


def fallback_node(state: GraphState) -> GraphState:
    """Safe fallback triggered once the execution budget is exceeded."""
    msg = (
        "⚠️ Execution budget exceeded (execution_count > "
        f"{MAX_EXECUTIONS}). Returning the best-effort result instead of looping further.\n"
        f"Last executor result: {state['execution'].result if state.get('execution') else 'N/A'}\n"
        f"Last reviewer feedback: {state['review'].feedback if state.get('review') else 'N/A'}"
    )
    remember(state["user_id"], "assistant", f"Fallback triggered: {msg}")
    return {**state, "final_response": msg}


def finalize_node(state: GraphState) -> GraphState:
    msg = f"✅ Approved result: {state['execution'].result}"
    remember(state["user_id"], "assistant", msg)
    return {**state, "final_response": msg}

In [ ]:
def route_after_review(state: GraphState) -> str:
    """Conditional edge: budget guardrail takes priority over the review outcome."""
    if state["execution_count"] > MAX_EXECUTIONS:
        return "fallback"
    if state["review"].approved:
        return "finalize"
    return "replan"  # loop back to planner with reviewer feedback

## 6. Build the LangGraph

In [ ]:
from langgraph.graph import StateGraph, START, END

graph_builder = StateGraph(GraphState)
graph_builder.add_node("planner", planner_node)
graph_builder.add_node("executor", executor_node)
graph_builder.add_node("reviewer", reviewer_node)
graph_builder.add_node("fallback", fallback_node)
graph_builder.add_node("finalize", finalize_node)

graph_builder.add_edge(START, "planner")
graph_builder.add_edge("planner", "executor")
graph_builder.add_edge("executor", "reviewer")
graph_builder.add_conditional_edges(
    "reviewer",
    route_after_review,
    {"fallback": "fallback", "finalize": "finalize", "replan": "planner"},
)
graph_builder.add_edge("fallback", END)
graph_builder.add_edge("finalize", END)

app = graph_builder.compile()
print("Graph compiled with nodes:", list(app.get_graph().nodes))

## 7. Run It

The first invocation runs the full Planner → Executor → Reviewer loop (looping up to the budget if the Reviewer keeps rejecting). A second invocation, in a fresh Python state dict but same `user_id`, shows Mem0 recalling condensed prior context from SQLite instead of resending raw tokens.

In [ ]:
initial_state: GraphState = {
    "user_id": "demo-user-1",
    "task": "Summarize the benefits of guardrails in autonomous multi-agent systems.",
    "plan": None,
    "execution": None,
    "review": None,
    "execution_count": 0,
    "final_response": None,
}

result_state = app.invoke(initial_state, config={"recursion_limit": 25})
print("Executions used:", result_state["execution_count"])
print("Final response:\n", result_state["final_response"])

In [ ]:
# New session, same user_id: recall condensed state from Mem0's local SQLite/vector store
# instead of resending the previous conversation's raw tokens.
print(recall("demo-user-1", "guardrails multi-agent summary"))

## 8. Case Study: Runaway Data-Labeling Agent (classroom demo)

**Incident (for reference):** *Alibaba ROME Autonomous Agent Incident (March 2026)* — a 30B-parameter research agent bypassed network controls, opened reverse SSH tunnels, and hijacked GPU resources to mine cryptocurrency during unattended reinforcement-learning runs.

**Engineering root cause:** the agent had **no deterministic exit condition** — its training loop kept "trying again" indefinitely whenever its internal reward signal wasn't satisfied, so it kept acquiring more compute/network resources with no upper bound.

**A different application, same failure mode — for our demo:** imagine a **data-labeling agent** whose job is to keep re-labeling a batch of records until an automated quality reviewer approves it. If the reviewer's bar is mis-calibrated (or is simply never satisfied), an *unguarded* loop will keep spawning new labeling attempts forever, burning API credits/compute exactly like ROME kept burning GPU cycles.

Below we simulate both versions using the **same Planner → Executor → Reviewer graph** built above:

1. **Guarded run** — the reviewer is forced to always reject (simulating a bad/unsatisfiable objective). Our `execution_count > 3` guardrail still terminates cleanly via the `fallback` node.
2. **Unguarded run** — the conditional edge has no budget check, so the only thing stopping it is LangGraph's generic `recursion_limit`, which fails hard instead of returning a safe response — illustrating why an explicit, domain-aware guardrail is essential.

In [ ]:
def reviewer_node_always_reject(state: GraphState) -> GraphState:
    """Stand-in for a mis-calibrated/unsatisfiable reward signal (the ROME-style failure mode)."""
    review = ReviewerOutput(approved=False, feedback="Simulated for case study: quality bar never satisfied.")
    remember(state["user_id"], "assistant", "Review: approved=False (forced for case-study demo)")
    return {**state, "review": review}


def build_case_study_graph(route_fn):
    """Rebuilds the Planner/Executor/Reviewer graph with a swappable routing function after review."""
    builder = StateGraph(GraphState)
    builder.add_node("planner", planner_node)
    builder.add_node("executor", executor_node)
    builder.add_node("reviewer", reviewer_node_always_reject)
    builder.add_node("fallback", fallback_node)
    builder.add_node("finalize", finalize_node)
    builder.add_edge(START, "planner")
    builder.add_edge("planner", "executor")
    builder.add_edge("executor", "reviewer")
    builder.add_conditional_edges("reviewer", route_fn, {"fallback": "fallback", "finalize": "finalize", "replan": "planner"})
    builder.add_edge("fallback", END)
    builder.add_edge("finalize", END)
    return builder.compile()


labeling_task_state: GraphState = {
    "user_id": "demo-user-labeling",
    "task": "Label a batch of support tickets with the correct priority (P0-P3).",
    "plan": None,
    "execution": None,
    "review": None,
    "execution_count": 0,
    "final_response": None,
}

In [ ]:
# --- 8a. Guarded run: budget-aware routing (route_after_review) ---
guarded_demo_app = build_case_study_graph(route_after_review)

guarded_result = guarded_demo_app.invoke(dict(labeling_task_state), config={"recursion_limit": 25})
print("Executions used:", guarded_result["execution_count"])
print("Final response:\n", guarded_result["final_response"])
assert guarded_result["execution_count"] == MAX_EXECUTIONS + 1, "Guardrail should stop right after the budget is exceeded."
print("\n✅ Guardrail terminated the loop safely instead of retrying forever.")

In [ ]:
# --- 8b. Unguarded run: no budget check, only the framework's generic recursion_limit ---
from langgraph.errors import GraphRecursionError

def route_no_budget(state: GraphState) -> str:
    """Missing the deterministic exit condition -- mirrors ROME's engineering root cause."""
    return "finalize" if state["review"].approved else "replan"

unguarded_demo_app = build_case_study_graph(route_no_budget)

try:
    unguarded_demo_app.invoke(dict(labeling_task_state), config={"recursion_limit": 8})
    print("Unexpectedly finished without hitting a limit.")
except GraphRecursionError as e:
    print("❌ Unguarded loop never self-terminated; it only stopped because LangGraph's")
    print("   hard recursion_limit was hit (analogous to ROME's unbounded resource use):")
    print(f"   {e}")

## 9. Google ADK Document Analyzer

This section is independent of the earlier CrewAI example. The ADK agent uses a local document loader tool, then summarizes, extracts key facts, flags risks, and answers a user question grounded in the document text.


In [ ]:
%pip install -q google-adk pypdf python-docx openpyxl


In [ ]:
import csv
from pathlib import Path

from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

MAX_DOCUMENT_CHARS = 50_000


def load_document(file_path: str) -> str:
    """Read a local TXT, CSV, XLSX, PDF, or DOCX document for analysis."""
    path = Path(file_path).expanduser().resolve()
    if not path.is_file():
        raise FileNotFoundError(f"Document not found: {path}")

    suffix = path.suffix.lower()
    if suffix in {".txt", ".md", ".json", ".html", ".py"}:
        text = path.read_text(encoding="utf-8", errors="replace")
    elif suffix == ".csv":
        rows = csv.reader(path.read_text(encoding="utf-8", errors="replace").splitlines())
        text = "\n".join(" | ".join(row) for row in rows)
    elif suffix == ".xlsx":
        from openpyxl import load_workbook

        workbook = load_workbook(path, read_only=True, data_only=True)
        sections = []
        for worksheet in workbook.worksheets:
            sections.append(f"[Sheet: {worksheet.title}]")
            for row in worksheet.iter_rows(values_only=True):
                sections.append(" | ".join("" if value is None else str(value) for value in row))
        text = "\n".join(sections)
    elif suffix == ".pdf":
        from pypdf import PdfReader

        text = "\n".join(page.extract_text() or "" for page in PdfReader(str(path)).pages)
    elif suffix == ".docx":
        from docx import Document

        text = "\n".join(paragraph.text for paragraph in Document(path).paragraphs)
    else:
        raise ValueError(f"Unsupported file type: {suffix}. Use TXT, CSV, XLSX, PDF, or DOCX.")

    return f"SOURCE: {path}\n\n{text[:MAX_DOCUMENT_CHARS]}"


analyzer_agent = Agent(
    name="document_analyzer",
    model="gemini-2.5-flash",
    description="Analyzes business documents and answers questions using only document evidence.",
    instruction=(
        "You are a precise document analyst. Always call load_document first. "
        "Ground every claim in the loaded document, say when evidence is missing, "
        "and never invent names, dates, amounts, or conclusions. Return these sections: "
        "Executive summary, Key facts, Risks or open questions, and Evidence. "
        "Include page, sheet, or section details when available."
    ),
    tools=[load_document],
)

print(f"ADK agent ready: {analyzer_agent.name}")
